In [12]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [13]:
import logging
import pandas as pd
import delta_sharing
from datetime import timezone

import general_functions.databricks_client as db_client
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import send_to_innkeepr_api_paginated

In [14]:
def timestamp_milliseconds(date):
    """
    Function to transform datesting in
    timestamp with milliseconds
    Args:
        date: date string
    """
    date = pd.to_datetime(date).replace(tzinfo=timezone.utc).timestamp() * 1000
    date = int(date)
    return date

In [15]:
from numpy import False_


customer = "PCS" #6617da9c01c2ab3bd12c63cb
date = "2026-08-06"#T15:00:00.000Z"
end_date = "2026-08-07"#T20:00:00.000Z"
use_session_id = False
url_tracking_param = "gclid"
url_campaign_param = "campaign_id"# klar_cpid"
url = return_api_url()
print(f"url = {url}")
account_id = return_workspace_ids()
account_id = [acc["id"] for acc in account_id if acc["name"] == customer]
account_id = account_id[0]

In [16]:
if use_session_id:
    profile_path = db_client.return_databricks_client()
    table_path = f"{profile_path}#delta_share_events.{account_id}.features_view_30_outlook"
    df = delta_sharing.load_as_pandas(table_path, limit=500000)
    #print(df["created"].min(), df["created"].max())
    #df = df[df["created"].astype("string")>=date]
    df = df[df["treatment"].isnull()]
    print(df["created"].min(), df["created"].max())
    session_ids=df["session"].unique().tolist()
    print(f"len session_ids = {len(session_ids)}")
    content = {"sessionId":session_ids[0:10]}
else:
    date_format_dash = "%Y-%m-%d"
    content = {
            "created": {
                "$gte":  pd.to_datetime(date).strftime(date_format_dash),
                "$lte": pd.to_datetime(end_date).strftime(date_format_dash),
            }
        }
    api_url = f"{url}api/sessions/query"
    sessions_response = send_to_innkeepr_api_paginated(
        api_url=api_url,
        accountID=account_id,
        content=content,
        logger=logging.getLogger("innkeepr")
    )
    sessions = pd.json_normalize(sessions_response)

In [17]:
sessions.to_parquet(f"DataChecks/sessions/test_sessions_{customer}_{date}_{end_date}.parquet")
print(sessions.shape)

In [18]:
campaing_columns = [col for col in sessions.columns if "campaign" in col]
campaing_columns

In [19]:
sessions.columns

In [20]:
cmapaign_but_no_gclid=sessions[(sessions[f"campaign.{url_tracking_param}"].isnull())&(sessions[f"campaign.{url_campaign_param}"].notnull())]
cmapaign_but_no_gclid[["id", f"campaign.{url_campaign_param}", f"campaign.{url_tracking_param}"]]

# Some Checks

## check signals

In [21]:
use_campaign_id = f"campaign.{url_campaign_param}" 
#use_campaign_id = f"campaign.gclid" 
print(use_campaign_id)
external_ids = sessions[use_campaign_id].dropna().unique().tolist()
print(len(external_ids))
signals = send_to_innkeepr_api_paginated(
    f"{url}api/ads/query",
    account_id,
    #{"externalId":["1435494216", "267300181"]},
    {"externalId": external_ids},
    #{"adAccountId":"552565349"},
    logging
)

signals = pd.json_normalize(signals)
signals

In [22]:
signals["externalId"].value_counts()

In [ ]:
signals["relates_to.treatment"].value_counts()

In [ ]:
treamtents = send_to_innkeepr_api_paginated(
    f"{url}api/treatments/query",
    account_id,
    
    {"id":signals["relates_to.treatment"].dropna().unique().tolist()},
    logging
)

treamtents = pd.json_normalize(treamtents)
treamtents

In [ ]:
for col in treamtents:
    for signal in ["1435494216", "267300181"]:
        temp = treamtents[treamtents[col].astype("str").str.contains(signal)]
        if len(temp)>0:
            print(col, signal, len(temp))

## check with conversions

In [ ]:
from general_functions.datetime_helper import transform_date_to_timestamp_milliseconds

test = sessions[sessions["campaign.ad_id"].isnull()==False]
meta_sessions = test["sessionId"].dropna().unique().tolist()
content = {
     "created": {
                     "$gte": transform_date_to_timestamp_milliseconds("20250818"),
                     "$lte": transform_date_to_timestamp_milliseconds("20250819"),
               }
    }
conversions = send_to_innkeepr_api_paginated(
    f"{url}/conversions/query",
    account_id,
    content,
    logging
    
)

conversions = pd.json_normalize(conversions)
conversions

In [ ]:
conversions_matches = conversions[conversions["sessionId"].isin(meta_sessions)]
conversions_matches

In [ ]:
externalIds = sessions[sessions["sessionId"].isin(conversions_matches["sessionId"])]["campaign.utm_id"].dropna().unique().tolist()
len(externalIds), externalIds

In [ ]:
signals = send_to_innkeepr_api_paginated(
    f"{url}/signals/query",
    account_id,
    {"externalId":externalIds},
    logging
)

signals = pd.json_normalize(signals)
signals

In [ ]:
treatments = send_to_innkeepr_api_paginated(
    f"{url}/treatments/query",
    account_id,
    {"id":signals["relates_to.treatment"].dropna().unique().tolist()},
    logging
)
treatments = pd.json_normalize(treatments)
treatments

# chech sessions and gclids

In [ ]:
sessions_with_conversions = pd.merge(sessions, conversions_matches, how="left", left_on="sessionId", right_on="sessionId")
sessions_with_gclid = sessions_with_conversions[sessions_with_conversions["campaign.gclid"].isnull() == False]